In [1]:
import numpy as np
from scipy.optimize import brentq
from scipy.special import erf
import matplotlib.pyplot as plt
from classy import Class

In [2]:
def y_of_z(z):
    return (1.0 + z) ** 1.5
 
def tanh_cdf(u):
    return 0.5 * (1.0 + np.tanh(u))
 

In [3]:
def triangle_cdf(u):
    """Symmetric triangular density on u in [-1, 1] (compact support -> finite duration)."""
    u = np.asarray(u, dtype=float)
    out = np.where(u < -1, 0.0,
          np.where(u < 0, 0.5 * (u + 1) ** 2,
          np.where(u < 1, 1 - 0.5 * (1 - u) ** 2, 1.0)))
    return out

In [4]:
def asym_triangle_cdf(u, r=2.0):
    """
    Asymmetric triangular density: left half-width 1, right half-width r
    (r>1 means the *rise* is slower / more extended than the completion, or
    vice versa depending on sign convention you adopt when building xe(z)).
    Peak height chosen so the density integrates to 1.
    """
    u = np.asarray(u, dtype=float)
    h = 2.0 / (1.0 + r)  # peak density at u=0
    out = np.where(u < -1, 0.0,
          np.where(u < 0, (h / 2.0) * (u + 1) ** 2,
          np.where(u < r, h / 2.0 + h * (u - 0.0) - (h / (2.0 * r)) * u ** 2, 1.0)))
    # renormalize numerically to guard against the piecewise algebra above
    return np.clip(out, 0.0, 1.0)

In [5]:
def erf_cdf(u):
    """Heaviside step convolved with a Gaussian -> erf. Lighter tails than tanh."""
    return 0.5 * (1.0 + erf(u / np.sqrt(2.0)))

In [6]:
KERNELS = {
    "tanh (baseline)": lambda u: tanh_cdf(u),
    "triangle":        lambda u: triangle_cdf(u),
    "asymmetric":      lambda u: asym_triangle_cdf(u, r=2.0),
    "erf (Gaussian-enveloped step)": lambda u: erf_cdf(u),
}

In [7]:
F_HE = 0.0813  # rough default consistent with Y_He ~ 0.245; replace with your run's value
 
def build_xe_table(cdf_func, y_star, w, z_grid):
    u = (y_star - y_of_z(z_grid)) / w
    xe = (1.0 + F_HE) * cdf_func(u)
    return z_grid, xe
 
# dense near the transition, sparse far away
Z_DENSE = np.linspace(3.0, 20.0, 25)
Z_WIDE = np.concatenate([np.linspace(0.0, 3.0, 4), Z_DENSE, np.linspace(20.0, 50.0, 6)])
Z_GRID = np.unique(Z_WIDE)
print(f"[debug] Z_GRID has {len(Z_GRID)} nodes")

[debug] Z_GRID has 33 nodes


In [8]:
TAU_TARGET = 0.054
 
common_settings = {
    "output": "tCl,pCl,lCl",
    "lensing": "yes",
    "l_max_scalars": 2500,
}
 
baseline = Class()
baseline.set(common_settings)
baseline.set({
    "reio_parametrization": "reio_camb",
    "tau_reio": TAU_TARGET,
})
baseline.compute()
 
z_reio_baseline = baseline.get_current_derived_parameters(["z_reio"])["z_reio"]
y_star_baseline = y_of_z(z_reio_baseline)
print(f"baseline z_reio = {z_reio_baseline:.3f}, y_star = {y_star_baseline:.3f}")

baseline z_reio = 7.639, y_star = 25.393


In [9]:
# default CLASS/CAMB width is ~0.5 in z near z_reio; convert roughly to a starting
# guess in y for the root-finder below
W_GUESS = 1.5 * np.sqrt(1.0 + z_reio_baseline) * 0.5

In [10]:
def get_tau_for_width(cdf_func, w, y_star=y_star_baseline, z_grid_snapshot=None):
    """
    z_grid_snapshot: pass an explicit, immutable copy of the z grid so this
    function never depends on whatever Z_GRID happens to be bound to in the
    notebook's global namespace at call time. This avoids the classic
    stale-state bug where re-running cells out of order in a Jupyter/VS Code
    interactive session leaves reio_inter_num and reio_inter_z built from
    two different versions of the grid.
    """
    if z_grid_snapshot is None:
        z_grid_snapshot = Z_GRID.copy()
    z_grid, xe = build_xe_table(cdf_func, y_star, w, z_grid_snapshot)
    z_list = [f"{z:.6f}" for z in z_grid]
    xe_list = [f"{x:.6f}" for x in xe]
    assert len(z_list) == len(xe_list) == len(z_grid), (
        f"grid/table length mismatch: z={len(z_list)}, xe={len(xe_list)}, "
        f"grid={len(z_grid)}"
    )
    n_nodes = len(z_list)
    cosmo = Class()
    cosmo.set(common_settings)
    cosmo.set({
        "reio_parametrization": "reio_inter",
        "reio_inter_num": n_nodes,
        "reio_inter_z": ",".join(z_list),
        "reio_inter_xe": ",".join(xe_list),
    })
    print(f"[debug] n_nodes={n_nodes}, reio_inter_z entries={len(z_list)}")
    cosmo.compute()
    tau = cosmo.get_current_derived_parameters(["tau_reio"])["tau_reio"]
    cosmo.struct_cleanup()
    cosmo.empty()
    return tau, cosmo

In [11]:
def solve_width(cdf_func, z_grid_snapshot, w_bracket=(0.3 * W_GUESS, 4.0 * W_GUESS)):
    f = lambda w: get_tau_for_width(cdf_func, w, z_grid_snapshot=z_grid_snapshot)[0] - TAU_TARGET
    w_solution = brentq(f, *w_bracket, xtol=1e-4)
    return w_solution

In [12]:
z_grid, xe = build_xe_table(cdf_func, y_star_baseline, W_GUESS, frozen_grid)
print(xe.min())

NameError: name 'cdf_func' is not defined

In [ ]:
widths = {}
runs = {"tanh (baseline)": baseline}
frozen_grid = Z_GRID.copy()  # single snapshot reused for every candidate, every iteration
 
for name, cdf_func in KERNELS.items():
    if name == "tanh (baseline)":
        continue
    w = solve_width(cdf_func, frozen_grid)
    widths[name] = w
    _, cosmo = get_tau_for_width(cdf_func, w, z_grid_snapshot=frozen_grid)
    runs[name] = cosmo
    print(f"{name}: w = {w:.4f}")

[debug] n_nodes=33, reio_inter_z entries=33


CosmoComputationError: 

Error in Class: thermodynamics_init(L:411) :error in thermodynamics_calculate_remaining_quantities(ppr,pba,pth,pvecback);
=>thermodynamics_calculate_remaining_quantities(L:1753) :error in thermodynamics_calculate_opticals(ppr,pth);
=>thermodynamics_calculate_opticals(L:3493) :condition (pth->thermodynamics_table[index_tau*pth->th_size+pth->index_th_dkappa] == 0.) is true; variation rate diverges

In [ ]:
def get_Dl(cosmo, spectrum, lmax=2500):
    cls = cosmo.lensed_cl(lmax)
    ell = cls["ell"]
    Tcmb_uK = cosmo.T_cmb() * 1e6
    Dl = ell * (ell + 1) / (2 * np.pi) * cls[spectrum] * Tcmb_uK ** 2
    return ell, Dl

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 10), sharex=False)
 
for name, cosmo in runs.items():
    ell, Dl_tt = get_Dl(cosmo, "tt")
    _, Dl_te = get_Dl(cosmo, "te")
    _, Dl_ee = get_Dl(cosmo, "ee")
    axes[0].plot(ell, Dl_tt, label=name)
    axes[1].plot(ell, Dl_te, label=name)
    axes[2].plot(ell[:30], Dl_ee[:30], label=name)  # low-l EE reionization bump
 
axes[0].set_title("TT")
axes[1].set_title("TE")
axes[2].set_title("EE, low-l reionization bump (l = 2-30)")
for ax in axes:
    ax.legend(fontsize=8)
    ax.set_xlabel("l")
    ax.set_ylabel(r"$D_l\ [\mu K^2]$")
axes[0].set_xscale("log")
axes[1].set_xscale("log")
 
plt.tight_layout()
plt.savefig("reio_kernel_comparison.png", dpi=150)
plt.show()

In [ ]:
#Cleanup

for cosmo in runs.values():
    cosmo.struct_cleanup()
    cosmo.empty()